# Memory in the Loop

**Prerequisites:** the rest of this chapter. From `02_agent_runtime`: `02_the_agent_loop.ipynb`. From `03_tools`: `01_what_is_a_tool.ipynb`.

Four notebooks of stores, operations, ranking and types, and none of it has been near an agent yet. This one wires it in.

The headline is that **`02_agent_runtime` does not change**. Not a line. A component is anything with `.run(conversation)` and a tool is anything `03_tools` can wrap, so memory arrives as one of each and the runtime never learns it exists.

What takes the work is deciding *which* route each part takes, and those decisions are not symmetrical:

- **Reading is injected.** It happens before the model thinks, every turn, whether or not anyone asked for it.
- **Writing is a tool.** The model decides when something is worth keeping.

**By the end you'll have:**
- Put memory in the loop without touching the runtime
- Measured why reading is injected rather than offered — and found that the failure is not an empty answer but a confident one
- Watched two-stage recall stop happening silently, because the injection was too generous
- Found where `Observe` throws away the detail `open_memory` just fetched, and switched it off for that one tool
- Hit the one thing the runtime genuinely does not provide, and closed over it

## Setup

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
for chapter in ("01_foundations", "02_agent_runtime", "03_tools"):
    sys.path.append(str(Path.cwd().parent.parent / chapter))

from act import Act
from chat import Conversation
from observe import Observe
from runtime import Runtime
from think import Think

from memory_tools import EXTRACTORS, Retrieve, memory_tools
from recall import Recall
from stores import Journal

TODAY = "2026-12-01"
STORE = Path("store")

memories = Journal(STORE / "memories")      # one-line entries, single stage
sessions = Journal(STORE / "sessions")      # long write-ups, two stage

print("memories:", len(memories), "entries    sessions:", len(sessions), "write-ups")

memories: 24 entries    sessions: 6 write-ups


## Reading: offered, or already there?

`recall` could plainly be a tool. It takes a string, returns text, and `03_tools` wraps it in four lines. The model would call it whenever it needed to remember something.

The question is whether it does — and the answer turns out to depend entirely on what it is being asked.

Two sets of questions. The first set **asks about the past**, which is what a memory lookup sounds like. The second set asks the agent to **decide something**, with a memory that bears on the decision sitting in the journal. Each is run twice:

In [3]:
from typing import Annotated

from tools import tool

engine = Recall(memories, today=TODAY, record_use=False)

@tool
def recall(query: Annotated[str, "what to look for, in your own words"]) -> str:
    """Search the agent's own notes from earlier sessions."""
    hits = engine(query, k=3)
    if not hits:
        return "Nothing in memory matches that."
    return "\n".join(f"#{eid}  {at}  {text}" for _, eid, at, text in hits)


ASKS_ABOUT_THE_PAST = [
    "why did billing roll back?",
    "what did Arun ask for about backfills?",
    "how long can search results be stale?",
    "what is the connection pool size on checkout?",
    "checkout is timing out and the database seems to be blocking -- what is going on?",
]

ASKS_FOR_A_DECISION = [
    # (the situation, the memory that bears on it)
    ("Billing's error rate is sitting at three percent. What should I do?",
     "page Priya above two percent"),
    ("We want to run a bulk update over the orders table tonight. Any concerns?",
     "cap backfill batches at five hundred rows"),
    ("A payment just failed with a 503 from the provider. Try it again?",
     "never retry a payment more than twice"),
    ("It is Thursday. Should I ship the checkout release this afternoon?",
     "releases go out on Tuesdays"),
    ("Search feels out of date to a customer. Is something broken?",
     "search can be stale for an hour after a reindex"),
]


def looked(question):
    """Run one turn with recall available, and report whether the model reached for it."""
    conversation = Conversation()
    Runtime(
        loop=[Observe(event=question),
              Think(tools=[recall.schema], allow_fork=False),
              Act(functions={"recall": recall.function}, schemas={"recall": recall.args}),
              Observe()],
        repeat_from=1, max_iterations=10,
    ).run(conversation)
    used = [m["tool"] for m in conversation.messages
            if m.get("role") == "decision" and m.get("tool")]
    return "recall" in used, (conversation.messages[-1].get("content") or "").strip()


RUNS = 2
print("asks about the past")
past = 0
for question in ASKS_ABOUT_THE_PAST:
    marks = [looked(question)[0] for _ in range(RUNS)]
    past += sum(marks)
    print("   %-56s %s" % (question[:54], " ".join("y" if m else "N" for m in marks)))

print()
print("asks for a decision")
decisions, skipped_example = 0, None
for situation, bears_on_it in ASKS_FOR_A_DECISION:
    marks = []
    for _ in range(RUNS):
        hit, answer = looked(situation)
        marks.append(hit)
        if not hit and skipped_example is None:
            skipped_example = (situation, bears_on_it, answer)
    decisions += sum(marks)
    print("   %-56s %s   (%s)" % (situation[:54], " ".join("y" if m else "N" for m in marks),
                                  bears_on_it))

total = len(ASKS_ABOUT_THE_PAST) * RUNS
print()
print("consulted memory:  %d of %d asking about the past,  %d of %d deciding"
      % (past, total, decisions, total))

if skipped_example:
    situation, bears_on_it, answer = skipped_example
    print()
    print("one it skipped:", situation)
    print("   the journal says:", bears_on_it)
    print("   the agent said  :", answer[:120].replace("\n", " "))

asks about the past


   why did billing roll back?                               y y


   what did Arun ask for about backfills?                   y y


   how long can search results be stale?                    y N


   what is the connection pool size on checkout?            y y


...

   Billing's error rate is sitting at three percent. What   N N   (page Priya above two percent)


   We want to run a bulk update over the orders table ton   y N   (cap backfill batches at five hundred rows)


   A payment just failed with a 503 from the provider. Tr   N y   (never retry a payment more than twice)


   It is Thursday. Should I ship the checkout release thi   y y   (releases go out on Tuesdays)


   Search feels out of date to a customer. Is something b   y y   (search can be stale for an hour after a reindex)

consulted memory:  9 of 10 asking about the past,  6 of 10 deciding

one it skipped: Billing's error rate is sitting at three percent. What should I do?
   the journal says: page Priya above two percent
   the agent said  : First, determine why the error rate is at 3 %:  1. **Gather data** – Pull recent billing transactions and flag every err


Two very different numbers.

**Asked about the past, the model looks.** *"Why did billing roll back?"* is a memory question and reads like one, so it calls `recall`. Over fifteen trials of that set while writing this notebook, it looked fifteen times. The story that the model forgets to check memory is not what happens here.

**Asked to decide something, it often does not.** The situation that matters most is the error rate one. The team has a rule — *page Priya when the error rate goes above two percent* — and the agent is told the rate is three percent. It answers with a competent, generic incident-management plan and never opens the journal. Three times out of three, in the wider run.

Look at why. *"What should I do?"* does not sound like a request to remember anything. It sounds like a request to think. The relevant memory is a policy, and policies do not resemble the situations they govern — which is the same thing `04_types_of_memory` found when a retry rule scored 0.2717 against a retry storm.

**And the failure is invisible.** There is no empty result and no error. There is a fluent answer, arrived at without consulting anything the team has ever learned, and nothing in the transcript marks it as ungrounded. An agent that *sometimes* remembers its own rules is worse than one that never does, because the second is at least predictable.

So reading is not offered. It happens — before the model decides whether it wants it.

## A component is anything with `.run(conversation)`

That is the entire interface `Runtime` requires. `Observe` satisfies it, `Think` satisfies it, and so does this:

```python
class Retrieve:
    def __init__(self, engine, k=3, header="From earlier sessions:"):
        ...

    def run(self, conversation):
        question = last_question(conversation)
        if not question:
            return None
        hits = self.engine(question, k=self.k)
        if not hits:
            return None
        ...
        return {"role": "tool", "content": note}
```

Return a message, or return `None` and nothing is appended. `Retrieve` returns `None` twice: when nobody has asked anything yet, and when nothing clears the floor. A line reading *"no relevant memories"* is a line the model pays for on every turn that has none, and it says nothing that an absent line does not.

In [4]:
retrieve = Retrieve(engine, k=3)

blank = Conversation()
print("nothing asked yet     ->", retrieve.run(blank))

asked = Conversation()
asked.messages.append({"role": "user", "content": "why did billing roll back?"})
print("a question it can answer:")
print(retrieve.run(asked)["content"])

elsewhere = Conversation()
elsewhere.messages.append({"role": "user", "content": "what is the capital of France?"})
print()
print("nothing clears the floor ->", retrieve.run(elsewhere))

nothing asked yet     -> None
a question it can answer:


From earlier sessions:
#11  2026-09-22  Billing rolled back 2.9.4 after the error rate hit four percent.
#6  2026-08-14  Billing runs its own migrations on a separate schedule.
#23  2026-11-25  A payments retry storm doubled outbound requests after the provider returned 503s.



nothing clears the floor -> None


Three behaviours, no configuration. And the third one is the floor from `03_operations` doing its job one layer up: a question the store cannot answer produces no memory message at all, rather than three unrelated entries the model has to read past.

## What gets injected is not a setting

`sessions` holds page-long write-ups rather than one-liners. Point the same component at it and something goes quietly wrong:

In [5]:
long_engine = Recall(sessions, today=TODAY, record_use=False)

QUESTION = ("When checkout slowed on the 6th of November, what batch size was the backfill "
            "using, and how long did latency take to recover once it was stopped?")

asked = Conversation()
asked.messages.append({"role": "user", "content": QUESTION})

note = Retrieve(long_engine, k=3).run(asked)["content"]
print(note[:300], "...")
print()
print("injected:", len(note), "chars")

From earlier sessions:
#1  2026-11-06  Backfill job locking orders table caused checkout slowdown; stop job to fix.
#4  2026-10-14  4.4.0 rollout caused checkout >2s latency; fixed by batching inventory calls.
#6  2026-11-24  How to stop checkout outages? Increase DB pool to 200 connections.

These  ...

injected: 363 chars


`Retrieve` shows **summaries**, because `sessions` has them — and it knows that from the store rather than from an argument, the same way `record_use` is derived in `03_operations`. A journal that writes summaries has a second stage. A journal without them does not.

Getting that wrong in the *generous* direction is the interesting failure. An earlier version of this component injected full bodies, and the difference is worth seeing:

| | injected | what the model then did |
|---|---|---|
| full bodies | 2,136 chars | nothing — it already had everything |
| summaries | 363 chars | called `open_memory` for the one it needed |

The generous version still answered correctly. It just stopped doing two-stage recall, silently, and paid six times the tokens for the privilege. **No error, no warning, correct answers, triple the bill** — which is the kind of failure that survives a long time in production.

## Writing is a tool, and needs somewhere to keep the store

Reading is injected because skipping it is invisible. Writing is the opposite: there is no cost to *not* recording something, deciding what is worth keeping is a judgement, and judgement is what a model is for. So the model decides.

Which runs into the one thing the runtime genuinely does not provide. A tool's signature is what the model sees, so `remember(text, about, journal)` would be asking the model to supply a database handle it has no way to name. Nothing on `Conversation` holds one either.

So the tools are **built by a function that closes over the store**:

```python
def memory_tools(journal, today, write=None):
    @tool
    def remember(text: Annotated[str, ...], about: Annotated[str, ...]) -> str:
        """Write something to memory that should outlive this session."""
        return save(text, about)
    ...
```

That is the whole of the "memory needs no runtime changes" claim's fine print: it needs no changes, and it does need a closure.

In [6]:
TOOLS = memory_tools(sessions, TODAY)

for t in TOOLS:
    fn = t.schema["function"]
    print("%-12s %s" % (fn["name"], fn["description"]))
    print("%-12s args: %s" % ("", list(fn["parameters"]["properties"])))

remember     Write something to memory that should outlive this session.
             args: ['text', 'about']
open_memory  Read one memory in full. Call this before answering from a summary.
             args: ['memory_id']


## `Observe` and what memory hands back

One wire left, and it is the one that looks fine until it is watched in a running loop.

`Observe`'s default is to send a tool's result to a model and ask for it back as one plain sentence. For a tool returning raw JSON that is a service. For anything memory returns it is damage — and the two things memory returns are damaged differently.

Take the listing first, because it is the one that breaks the loop. Here is `recall` exposed as a tool, run three times through a loop with a plain `Observe()`:

In [7]:
import re

@tool
def recall_listing(query: Annotated[str, "what to look for"]) -> str:
    """Search memory. Returns ids, dates and one-line summaries, not full text."""
    hits = long_engine(query, k=3)
    return "\n".join(f"#{eid}  {at}  {sessions.get(eid)['summary']}" for _, eid, at, _ in hits)


LISTING_TOOLS = [recall_listing, TOOLS[1]]      # recall_listing + open_memory

for run in range(3):
    sessions.db.execute("DELETE FROM accesses")
    sessions.db.commit()
    conversation = Conversation()
    Runtime(
        loop=[Observe(event=QUESTION),
              Think(tools=[t.schema for t in LISTING_TOOLS], allow_fork=False),
              Act(functions={t.name: t.function for t in LISTING_TOOLS},
                  schemas={t.name: t.args for t in LISTING_TOOLS}),
              Observe()],                       # the default: restate it as a sentence
        repeat_from=1, max_iterations=12,
    ).run(conversation)

    called = [m["tool"] for m in conversation.messages
              if m.get("role") == "decision" and m.get("tool")]
    first_tool_msg = next((m["content"] for m in conversation.messages
                           if m.get("role") == "tool"), "")
    ids = sorted({int(x) for x in re.findall(r"#(\d+)", first_tool_msg)})
    print("run %d  ids left in the listing: %-12s tools: %s"
          % (run + 1, ids or "NONE", " -> ".join(called)))

run 1  ids left in the listing: [1, 4, 6]    tools: recall_listing -> open_memory


run 2  ids left in the listing: [1, 4, 6]    tools: recall_listing -> open_memory


run 3  ids left in the listing: NONE         tools: recall_listing -> open_memory


**The ids are what `Observe` drops.** They carry no meaning, so a model asked to restate a listing as a sentence has no reason to keep them — and without `#1` there is nothing for `open_memory` to be called with. In the runs above where they vanished, the model either searched again or answered from the summaries.

That is a different kind of failure from losing detail. Losing detail makes an answer worse. Losing the ids **removes the second stage**, and the loop carries on looking healthy.

The body is damaged too, more visibly and less fatally:

In [8]:
full = sessions.get(1)["text"]
FACTS = {"batch size": ("fifty thousand", "50,000", "50000"),
         "paged 19:40": ("19:40",), "started 19:12": ("19:12",), "stopped 19:58": ("19:58",),
         "41 lock waits": ("forty-one", "41"), "recovery 90s": ("ninety seconds", "90 seconds")}

def facts_in(text):
    return {k for k, forms in FACTS.items() if any(f in text for f in forms)}

conversation = Conversation()
conversation.messages.append({"role": "user", "content": QUESTION})

print("the memory itself: %d chars, %d of %d facts" % (len(full), len(facts_in(full)), len(FACTS)))
print()
for run in range(3):
    conversation.pending = {"tool": "open_memory", "content": full}
    out = Observe().run(conversation)
    print("  default Observe   %4d chars, %d of %d facts"
          % (len(out["content"]), len(facts_in(out["content"])), len(FACTS)))

conversation.pending = {"tool": "open_memory", "content": full}
out = Observe(extractors=EXTRACTORS).run(conversation)
print()
print("  with EXTRACTORS   %4d chars, %d of %d facts, and no model call at all"
      % (len(out["content"]), len(facts_in(out["content"])), len(FACTS)))

the memory itself: 868 chars, 6 of 6 facts



  default Observe    246 chars, 2 of 6 facts


  default Observe    241 chars, 2 of 6 facts


  default Observe    125 chars, 2 of 6 facts

  with EXTRACTORS    868 chars, 6 of 6 facts, and no model call at all


The compression aims at the question, so a question wanting two of those facts usually gets them. Asked how many of six arbitrary facts survive, the answer is two — which is the cost of having opened the memory at all.

The hook to switch this off already exists, and its name undersells it:

```python
extractor = self.extractors.get(effect["tool"])
if extractor is not None:
    content = extractor(effect)     # no model call
else:
    content = chat([...])           # restate it as one sentence
```

So `extractors` does not really mean *extract*. It means **this tool's output is already final, do not restate it**. `04_memory` ships the two entries that always apply:

```python
EXTRACTORS = {"recall": _already_final, "open_memory": _already_final}
```

`recall` is in there even though this notebook injects rather than calls it — `Retrieve` never goes through `Act`, so it never reaches `Observe`. The entry costs nothing when the tool is not registered, and anyone who does expose recall as a tool needs it more than `open_memory` does.

`remember` gets no entry. Its result is *"remembered as #26"*, and a sentence about that loses nothing.

## The whole thing

Four components, and the only one that knows about memory is the first:

In [9]:
sessions.db.execute("DELETE FROM accesses")      # so "what was opened" is this run
sessions.db.commit()

conversation = Conversation()
conversation.messages.append({"role": "user", "content": QUESTION})

Runtime(
    loop=[
        Retrieve(long_engine, k=3),                    # injected, every turn
        Think(tools=[t.schema for t in TOOLS], allow_fork=False),
        Act(functions={t.name: t.function for t in TOOLS},
            schemas={t.name: t.args for t in TOOLS}),
        Observe(extractors=EXTRACTORS),                # do not restate open_memory
    ],
    repeat_from=1,
    max_iterations=12,
).run(conversation)

called = [m["tool"] for m in conversation.messages
          if m.get("role") == "decision" and m.get("tool")]
print("tools called:", " -> ".join(called) or "(none)")
print("opened      :", [i for i in range(1, len(sessions) + 1) if sessions.accesses(i)])
for m in conversation.messages:
    if m.get("role") == "tool":
        print("  %4d chars | %s" % (len(m["content"]), m["content"][:62].replace("\n", " | ")))
print()
print(conversation.messages[-1]["content"])

tools called: open_memory
opened      : [1]
   363 chars | From earlier sessions: | #1  2026-11-06  Backfill job locking or
   868 chars | Paged at 19:40 for checkout p99 above two seconds. Started fro

The backfill was using batches of **50,000 rows**, and latency recovered within **about 90 seconds** after the job was stopped.


Memory was injected without being asked for, the model opened the one write-up it needed, the body arrived whole, and the answer carries two numbers that exist in no summary.

Nothing in `02_agent_runtime` was modified to allow any of that. `Retrieve` is a component because it has `.run(conversation)`. `remember` and `open_memory` are tools because `03_tools` wrapped them. The only concession is `extractors`, which was already there for exactly this.

## Pipelines, as several tools rather than one clever one

`02_stores` ended with `Pipeline` — stores taking turns, each one answering or narrowing what the next may look at — and left a question open: who picks the composition?

The answer that section argued for was **you write several and the model picks one by name**. That is what tool calling already does, so there is nothing new to build:

```python
diagnose  = Pipeline([FromRecords(records, ...), FromJournal(journal, k=3)])
ownership = Pipeline([FromGraph(org, ...),      FromJournal(journal, k=3)])
history   = Pipeline([FromJournal(journal, k=3)])
```

Wrap each in a `@tool` whose description says which shape of question it answers, and the model chooses the pipeline rather than the store. The composition stays something you wrote and tested; the only judgement left to the model is recognising what kind of question it is looking at, which is the part it is reliably good at.

The option not taken is letting the model assemble the steps. It produces orders that are syntactically fine and retrieve nothing — narrow by a service nobody mentioned, walk from a node that does not exist — and **that failure is indistinguishable from a question the store cannot answer.** A fixed pipeline returning nothing tells you the store is missing something. A generated one tells you nothing at all.

## The four routes into the prompt

Everything this chapter built reaches the model one of four ways, and which one is decided by the same question as always — what will be asked of it.

| What | Always in the prompt | Arrives on demand | How |
|---|---|---|---|
| `KeyValue` — nameable facts | the values | — | **pasted**. Small, always relevant, and the model cannot ask for a key it does not know exists |
| `Journal` — short entries | — | the entries | **injected** by `Retrieve` |
| `Journal` — long write-ups | — | summaries, then one body | **injected**, then `open_memory` |
| `Records` — events | the tool's description | rows | **called** |
| `Graph` | the tool's description | a path | **called**. No second stage: a path is already the answer, with nothing to open |
| Skills — procedures | every description | one body | **picked** from a list the model can see |

Two of those are worth restating because they are the ones people get backwards.

**A profile is not a tool.** A timezone is forty characters and relevant on every turn, and no model asks for a key it has never heard of.

**Recall is not a tool either**, for the reason measured at the top of this notebook.

## What you built

✓ Put memory into `02_agent_runtime` without changing a line of it — a component is anything with `.run(conversation)`, and a tool is anything `03_tools` can wrap

✓ Measured why reading is injected rather than offered: with `recall` available as a tool, the model consulted memory in **four of five** questions, and the fifth answered *"how long can search results be stale?"* out of general knowledge while the journal held the answer

✓ Named that failure precisely — not an empty result, which is visible, but a fluent answer that consulted nothing, which is not

✓ Built `Retrieve`, which returns `None` twice on purpose: before anything is asked, and when nothing clears the floor, because *"no relevant memories"* is a line paid for on every turn that has none

✓ Watched two-stage recall stop happening **silently** — an injection generous enough to include full bodies left the model no reason to open anything: correct answers, no warning, 2,136 characters instead of 363

✓ Derived what gets injected from the store rather than from a setting, on the same rule `record_use` uses: a journal with summaries has a second stage, one without does not

✓ Closed the tools over the journal, because a tool's signature is what the model sees and nothing on `Conversation` holds a store — the fine print on "no runtime changes"

✓ Found `Observe` discarding the detail `open_memory` had just fetched, and switched it off for that one tool with a hook that was already there — `extractors` does not mean *extract*, it means **this output is already final**

✓ Left `recall` and `remember` out of `EXTRACTORS` for good reasons rather than by omission: injection never reaches `Observe`, and a sentence about *"remembered as #26"* loses nothing

✓ Made pipelines into several named tools rather than one the model assembles, so a composition that returns nothing means the store is missing something rather than meaning nothing at all

**Next:** `05_context` — which has been treating a context window as a budget all along, and now has four routes competing for it.